# Large Code vs Graph Risk Analysis

This notebook checks whether large Java methods in `data.jsonl` actually become large/risky graphs after pipeline 01.

Important: do not filter methods only because they are over 50 lines. A 70-line method may produce a small graph, while a 20-line method with dense expressions may produce a bigger graph. Use this notebook to separate:

- Large code but graph looks safe.
- Large code and graph is large/risky.
- Large code with missing/empty CFG/DDG/PDG, which may indicate extraction problems.

Nothing here deletes files. Optional CSV export is disabled by default.

In [1]:
from pathlib import Path
import json
import math
import statistics

import pandas as pd

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT.name != "Spectral-Software" and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent

OUTPUT_ROOT = PROJECT_ROOT.parent / "outputs"

DATASETS = {
    "type1": {
        "data_jsonl": PROJECT_ROOT / "bench_data" / "bcb_full_type1" / "data.jsonl",
        "features_dir": OUTPUT_ROOT / "type1" / "dataset_features",
    },
    "type2": {
        "data_jsonl": PROJECT_ROOT / "bench_data" / "bcb_full_type2" / "data.jsonl",
        "features_dir": OUTPUT_ROOT / "type2" / "dataset_features",
    },
}

# You asked about >50 lines. Keep several buckets so we can see the tail too.
LINE_THRESHOLDS = [50, 100, 200, 500, 1000]
CHAR_THRESHOLDS = [5_000, 10_000, 25_000, 50_000]

# Tune these based on memory and spectral runtime.
RISK_MAX_NODES = 2_000
RISK_MAX_EDGES = 10_000

# Keep this True for speed: only read dataset_features JSON files for large-code candidates.
LIMIT_GRAPH_LOAD_TO_LARGE_CODE = True
LARGE_CODE_LINE_THRESHOLD = 50

GRAPH_TYPES = ["ast", "cfg", "ddg", "pdg", "cpg"]
DATASETS

{'type1': {'data_jsonl': WindowsPath('c:/Users/koush/PyProjects/Spectral-Software/bench_data/bcb_full_type1/data.jsonl'),
  'features_dir': WindowsPath('c:/Users/koush/PyProjects/outputs/type1/dataset_features')},
 'type2': {'data_jsonl': WindowsPath('c:/Users/koush/PyProjects/Spectral-Software/bench_data/bcb_full_type2/data.jsonl'),
  'features_dir': WindowsPath('c:/Users/koush/PyProjects/outputs/type2/dataset_features')}}

## Load Code Size Metrics

This scans `data.jsonl` row by row and records source-code size metrics.

In [2]:
def load_code_sizes(dataset_name: str, data_jsonl: Path) -> pd.DataFrame:
    rows = []
    with data_jsonl.open("r", encoding="utf-8") as f:
        for row_no, line in enumerate(f, start=1):
            record = json.loads(line)
            idx = str(record["idx"])
            code = record.get("func", "") or ""
            lines = code.splitlines() or [""]
            rows.append({
                "dataset": dataset_name,
                "idx": idx,
                "row_no": row_no,
                "chars": len(code),
                "lines": len(lines),
                "longest_line": max(len(x) for x in lines),
                "brace_count": code.count("{") + code.count("}"),
                "semicolon_count": code.count(";"),
            })
    return pd.DataFrame(rows)


code_size_df = pd.concat(
    [load_code_sizes(name, cfg["data_jsonl"]) for name, cfg in DATASETS.items()],
    ignore_index=True,
)

code_size_df.head()

,dataset,idx,row_no,chars,lines,longest_line,brace_count,semicolon_count
0,type1,118336,1,58,3,32,2,1
1,type1,117624,2,94,3,58,2,1
2,type1,117722,3,144,3,80,2,1
3,type1,117806,4,2889,65,343,28,42
4,type1,118127,5,1621,27,187,8,21


## Count Large Code Methods

In [3]:
summary_rows = []
for dataset_name, group in code_size_df.groupby("dataset"):
    row = {
        "dataset": dataset_name,
        "methods": len(group),
        "mean_lines": group["lines"].mean(),
        "p50_lines": group["lines"].quantile(0.50),
        "p95_lines": group["lines"].quantile(0.95),
        "p99_lines": group["lines"].quantile(0.99),
        "max_lines": group["lines"].max(),
        "max_chars": group["chars"].max(),
    }
    for threshold in LINE_THRESHOLDS:
        row[f"lines_gt_{threshold}"] = int((group["lines"] > threshold).sum())
        row[f"lines_gt_{threshold}_rate"] = float((group["lines"] > threshold).mean())
    for threshold in CHAR_THRESHOLDS:
        row[f"chars_gt_{threshold}"] = int((group["chars"] > threshold).sum())
        row[f"chars_gt_{threshold}_rate"] = float((group["chars"] > threshold).mean())
    summary_rows.append(row)

large_code_summary = pd.DataFrame(summary_rows).set_index("dataset")
large_code_summary

,methods,mean_lines,p50_lines,p95_lines,p99_lines,max_lines,max_chars,lines_gt_50,lines_gt_50_rate,lines_gt_100,...,lines_gt_1000,lines_gt_1000_rate,chars_gt_5000,chars_gt_5000_rate,chars_gt_10000,chars_gt_10000_rate,chars_gt_25000,chars_gt_25000_rate,chars_gt_50000,chars_gt_50000_rate
dataset,,,,,,,,,,,,,,,,,,,,,
type1,107413,9.594667,4.0,32.0,80.0,2003,107178,2475,0.023042,644,...,4,0.000037,501,0.004664,122,0.001136,18,0.000168,7,0.000065
type2,191732,8.913442,4.0,29.0,68.0,2003,107178,3443,0.017957,819,...,5,0.000026,772,0.004026,217,0.001132,36,0.000188,9,0.000047


## Load Graph Size Metrics From Pipeline 01

If `dataset_features` exists, this joins graph sizes to code sizes. If you have not rerun pipeline 01 after fixing Joern overlays, rerun it first; old CFG/DDG/PDG sizes may be invalid.

In [4]:
def load_graph_sizes(dataset_name: str, features_dir: Path, candidate_ids: set[str] | None = None) -> pd.DataFrame:
    rows = []
    if not features_dir.exists():
        return pd.DataFrame(rows)

    paths = [features_dir / f"{idx}.json" for idx in sorted(candidate_ids)] if candidate_ids is not None else features_dir.glob("*.json")

    for path in paths:
        if not path.exists():
            continue
        try:
            record = json.loads(path.read_text(encoding="utf-8"))
        except Exception:
            continue

        idx = str(record.get("idx", path.stem))
        row = {"dataset": dataset_name, "idx": idx}
        for graph_type in GRAPH_TYPES:
            graph = record.get("features", {}).get(graph_type, {})
            row[f"{graph_type}_nodes"] = int(graph.get("nodes", 0) or 0)
            row[f"{graph_type}_edges"] = int(graph.get("edges", 0) or 0)
        rows.append(row)

    return pd.DataFrame(rows)


candidate_ids_by_dataset = None
if LIMIT_GRAPH_LOAD_TO_LARGE_CODE:
    candidate_ids_by_dataset = {
        dataset_name: set(group.loc[group["lines"] > LARGE_CODE_LINE_THRESHOLD, "idx"].astype(str))
        for dataset_name, group in code_size_df.groupby("dataset")
    }
    print("Fast mode enabled: loading graph JSON only for large-code candidates.")
    print({k: len(v) for k, v in candidate_ids_by_dataset.items()})

graph_size_df = pd.concat(
    [
        load_graph_sizes(
            name,
            cfg["features_dir"],
            candidate_ids_by_dataset.get(name) if candidate_ids_by_dataset else None,
        )
        for name, cfg in DATASETS.items()
    ],
    ignore_index=True,
)

print(f"Loaded graph sizes for {len(graph_size_df):,} methods.")
graph_size_df.head()

KeyboardInterrupt: 

In [ ]:
if graph_size_df.empty:
    analysis_df = code_size_df.copy()
    print("No graph-size data loaded. Run pipeline 01 first if you want code-vs-graph analysis.")
else:
    analysis_df = code_size_df.merge(graph_size_df, on=["dataset", "idx"], how="left")
    node_cols = [f"{g}_nodes" for g in GRAPH_TYPES]
    edge_cols = [f"{g}_edges" for g in GRAPH_TYPES]
    analysis_df["max_graph_nodes"] = analysis_df[node_cols].max(axis=1)
    analysis_df["max_graph_edges"] = analysis_df[edge_cols].max(axis=1)
    analysis_df["missing_graph_metrics"] = analysis_df[node_cols].isna().any(axis=1)
    analysis_df["risky_graph"] = (
        (analysis_df["max_graph_nodes"] > RISK_MAX_NODES)
        | (analysis_df["max_graph_edges"] > RISK_MAX_EDGES)
    )
    analysis_df["empty_cfg"] = analysis_df["cfg_nodes"].fillna(0).eq(0)
    analysis_df["tiny_ddg"] = analysis_df["ddg_nodes"].fillna(0).le(3)
    analysis_df["tiny_pdg"] = analysis_df["pdg_nodes"].fillna(0).le(3)

analysis_df["large_code_gt_50_lines"] = analysis_df["lines"] > 50
analysis_df.head()

## Big Code But Graph Safe vs Risky

In [ ]:
if "risky_graph" in analysis_df.columns:
    bucket_summary = (
        analysis_df
        .assign(
            big_code=lambda d: d["lines"] > 50,
            graph_safe=lambda d: ~d["risky_graph"],
        )
        .groupby(["dataset", "big_code", "graph_safe"])
        .size()
        .reset_index(name="methods")
    )
    display(bucket_summary)
else:
    print("Graph metrics are not available yet.")

## Candidate Tables

Interpretation:

- `large_but_graph_safe`: big source code, but graph sizes are under risk thresholds. Usually keep these.
- `large_and_graph_risky`: big source code and large graph. These are candidates for manual inspection or filtered benchmark variants.
- `graph_risky_even_if_short`: not necessarily big source code, but graph is large. These are easy to miss if we only filter by line count.

In [ ]:
base_cols = ["dataset", "idx", "lines", "chars", "longest_line", "brace_count", "semicolon_count"]
graph_cols = [c for c in analysis_df.columns if c.endswith("_nodes") or c.endswith("_edges")]
display_cols = base_cols + ["max_graph_nodes", "max_graph_edges"] + graph_cols

if "risky_graph" in analysis_df.columns:
    large_but_graph_safe = analysis_df[(analysis_df["lines"] > 50) & (~analysis_df["risky_graph"])].copy()
    large_and_graph_risky = analysis_df[(analysis_df["lines"] > 50) & (analysis_df["risky_graph"])].copy()
    graph_risky_even_if_short = analysis_df[(analysis_df["lines"] <= 50) & (analysis_df["risky_graph"])].copy()

    print("large_but_graph_safe:", len(large_but_graph_safe))
    display(large_but_graph_safe.sort_values(["lines", "max_graph_nodes"], ascending=False)[display_cols].head(30))

    print("large_and_graph_risky:", len(large_and_graph_risky))
    display(large_and_graph_risky.sort_values(["max_graph_nodes", "max_graph_edges"], ascending=False)[display_cols].head(30))

    print("graph_risky_even_if_short:", len(graph_risky_even_if_short))
    display(graph_risky_even_if_short.sort_values(["max_graph_nodes", "max_graph_edges"], ascending=False)[display_cols].head(30))
else:
    largest_code_only = analysis_df.sort_values(["lines", "chars"], ascending=False)
    display(largest_code_only[base_cols].head(50))

## Check For Extraction Problems In Large Code

After the Joern overlay fix, CFG should usually be non-empty for methods with executable bodies. If many large methods still have empty CFG/tiny DDG/tiny PDG, inspect pipeline 01 logs or the Java wrapper.

In [ ]:
if {"empty_cfg", "tiny_ddg", "tiny_pdg"}.issubset(analysis_df.columns):
    issue_summary = (
        analysis_df[analysis_df["lines"] > 50]
        .groupby("dataset")[["empty_cfg", "tiny_ddg", "tiny_pdg"]]
        .agg(["sum", "mean"])
    )
    display(issue_summary)

    suspicious_large = analysis_df[
        (analysis_df["lines"] > 50)
        & (analysis_df["empty_cfg"] | analysis_df["tiny_ddg"] | analysis_df["tiny_pdg"])
    ].copy()
    print("suspicious_large:", len(suspicious_large))
    display(suspicious_large.sort_values(["lines", "chars"], ascending=False)[display_cols + ["empty_cfg", "tiny_ddg", "tiny_pdg"]].head(50))
else:
    print("Graph issue metrics are not available yet.")

## Optional: Save Review CSVs

This writes CSVs under `bench_data/large_code_reports`. Keep `SAVE_REPORTS=False` unless you want files.

In [ ]:
SAVE_REPORTS = False

if SAVE_REPORTS:
    out_dir = PROJECT_ROOT / "bench_data" / "large_code_reports"
    out_dir.mkdir(parents=True, exist_ok=True)
    large_code_summary.to_csv(out_dir / "large_code_summary.csv")
    analysis_df.to_csv(out_dir / "large_code_vs_graph_all_methods.csv", index=False)
    if "risky_graph" in analysis_df.columns:
        large_but_graph_safe.to_csv(out_dir / "large_but_graph_safe.csv", index=False)
        large_and_graph_risky.to_csv(out_dir / "large_and_graph_risky.csv", index=False)
        graph_risky_even_if_short.to_csv(out_dir / "graph_risky_even_if_short.csv", index=False)
    print(f"Saved reports to {out_dir}")
else:
    print("SAVE_REPORTS is False. No files written.")

## Recommendation

Do not ignore methods just because they are over 50 lines. If the graph is safe, keep them. If the graph is too large or extraction is broken, create a separate filtered benchmark variant and record the filter rule in metadata. That way the main BCB-derived benchmark remains reproducible.